# 8.7 — Mini-Batch and the Cost of Sampling

**Chapter 8, section 8.1.4**, and the starting point for **Exercise 3** and **Exercise 5**.

**The question this notebook answers:** if a mini-batch is a twentieth of the training set, is the
iteration a twentieth of the cost?

It is not, and the reason is stated in one sentence of the Spark optimization guide: *"Sampling
still requires a pass over the entire RDD, so decreasing `miniBatchFraction` may not speed up
optimization much."* An iteration costs two separable things — the **scan** of the distributed
dataset and the **arithmetic** on the records that survive the sample — and the mini-batch shrinks
only the second. This notebook measures both, separately, and then shows the case where
mini-batching does pay.

It also carries the accumulator the rest of the chapter uses: the triple of a summed gradient, a
summed loss, and a count. The last section is about why the count is in there.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-8.7")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. The training set

Ten thousand labelled points, split ninety-ten, giving roughly nine thousand training examples.
A mini-batch of 512 from nine thousand is a sampling fraction of about $512/9000 \approx 0.057$ —
that is, each iteration draws roughly one record in eighteen.

In [2]:
from sklearn.datasets import make_blobs

n = 10000
X, y = make_blobs(n_samples=n, centers=2, n_features=2, cluster_std=0.5, random_state=2)

rdd_X = sc.parallelize(X)
rdd_y = sc.parallelize(y)
traindata, testdata = rdd_y.zip(rdd_X).randomSplit([0.9, 0.1], seed=12345)
traindata = traindata.map(lambda r: (float(r[0]), np.append(r[1], 1.0)))
traindata.cache()
train_size = traindata.count()          # computed once, on the driver, not inside the loop

MINI_BATCH = 512
fraction = MINI_BATCH / train_size
print(f"training examples : {train_size}")
print(f"mini-batch size   : {MINI_BATCH}")
print(f"sampling fraction : {fraction:.4f}   (about 1 record in {1/fraction:.0f})")

training examples : 8959
mini-batch size   : 512
sampling fraction : 0.0571   (about 1 record in 17)


---

## 2. The iteration

The loop below is the one the chapter prints. Three things are worth naming in it.

**The sample.** `traindata.sample(False, fraction, 1 + i)` draws the mini-batch. The seed is
`1 + i`, which changes with the iteration — a fixed seed would draw the *same* records every time
and the run would not be stochastic at all.

**The accumulator.** `treeAggregate` starts from a triple: a zero gradient vector, a zero loss, and
a zero count. `seqOp` folds in one training example, `combOp` merges two accumulators by
componentwise addition. Because that merge is associative, Spark may combine the partial results
in any tree shape on the executors and send the driver a single triple.

**The division, and where it happens.** The aggregation returns a **sum** and a **count**, never an
average. The division by `res[2]` happens on the driver, after the reduction has finished.
Section 5 is about why that is not a stylistic choice.

The two lines after it add the $\ell_2$ penalty to the objective and to the gradient. Neither
touches the data, because the penalty depends only on the parameter vector, which is on the driver
already.

In [3]:
def iteration(parameter_vector, i, regularization=0.01, learningRate=0.01,
              mini_batch_size=MINI_BATCH, seq_op=None):
    parameter_size = len(parameter_vector)
    bc_weights = parameter_vector

    min_batch = traindata.sample(False, mini_batch_size / train_size, 1 + i)

    res = min_batch.treeAggregate(
          (np.zeros(parameter_size), 0, 0),
          seq_op if seq_op is not None else
          (lambda x, y: (x[0]
                       + (y[1]) * (-y[0] + (1 / (np.exp(-np.dot(y[1], bc_weights)) + 1))),
                         x[1]
                       + y[0] * (-(np.dot(y[1], bc_weights)))
                       + np.log(1 + np.exp(np.dot(y[1], bc_weights))),
                         x[2] + 1)),
          lambda x, y: (x[0] + y[0], x[1] + y[1], x[2] + y[2]))

    cost = res[1] + regularization * (np.square(parameter_vector).sum())
    gradient_derivative = (1.0 / res[2]) * res[0] + 2 * regularization * parameter_vector

    parameter_vector = parameter_vector - learningRate * gradient_derivative
    return parameter_vector, cost, res[2]


np.random.seed(0)
beta = np.random.normal(0, 0.1, len(traindata.first()[1]))
for i in range(5):
    beta, cost, seen = iteration(beta, i)
    print(f"iteration {i}  gradients computed on {seen:5} records  cost = {cost:.3f}")

iteration 0  gradients computed on   523 records  cost = 279.371
iteration 1  gradients computed on   506 records  cost = 255.990
iteration 2  gradients computed on   522 records  cost = 255.071
iteration 3  gradients computed on   515 records  cost = 244.836
iteration 4  gradients computed on   504 records  cost = 229.756


---

## 3. What the scan costs, measured

The claim to test is that the scan does not shrink with the batch. Two accumulators measure it in
a single job: one counts every record that **enters** the pipeline, the other counts every record
whose **gradient is computed**.

> **What this instrument can and cannot see.** Spark accumulators updated inside a transformation
> are only exact when the transformation runs exactly once. A recomputed or retried partition would
> be counted twice. The RDD here is cached and nothing fails, so these numbers are exact — but the
> guarantee comes from those two facts, not from the accumulator.

In [4]:
def scan_versus_batch(mini_batch_size, seed=1):
    scanned = sc.accumulator(0)
    sampled = sc.accumulator(0)

    def count_scanned(r):
        scanned.add(1)
        return r

    def count_sampled(r):
        sampled.add(1)
        return r

    (traindata.map(count_scanned)
              .sample(False, mini_batch_size / train_size, seed)
              .map(count_sampled)
              .count())
    return scanned.value, sampled.value


import pandas as pd

rows = []
for mb in (train_size, 4096, 512, 64):
    scanned, sampled = scan_versus_batch(mb)
    rows.append({"nominal batch": mb,
                 "miniBatchFraction": round(mb / train_size, 4),
                 "records scanned": scanned,
                 "gradients computed": sampled})
print(pd.DataFrame(rows).to_string(index=False))

 nominal batch  miniBatchFraction  records scanned  gradients computed
          8959             1.0000             8959                8959
          4096             0.4572             8959                4096
           512             0.0571             8959                 523
            64             0.0071             8959                  68


The right-hand column falls by a factor of more than a hundred from top to bottom. The one beside
it does not move at all. Every iteration reads every record, whatever fraction it keeps, because
`sample` is a filter and a filter has to look at what it is filtering.

So the arithmetic per iteration is proportional to `miniBatchFraction`, and the scan is not. What
that does to the clock depends entirely on which of the two dominates.

In [5]:
import time

def time_iterations(mini_batch_size, repeats=7, **kw):
    np.random.seed(0)
    b = np.random.normal(0, 0.1, len(traindata.first()[1]))
    b, _, _ = iteration(b, 0, mini_batch_size=mini_batch_size, **kw)   # warm-up, not timed
    secs = []
    for i in range(repeats):
        t0 = time.perf_counter()
        b, _, _ = iteration(b, i, mini_batch_size=mini_batch_size, **kw)
        secs.append(time.perf_counter() - t0)
    return float(np.median(secs))


measured = {mb: time_iterations(mb) for mb in (train_size, 4096, 512, 64)}
full = measured[train_size]

rows = [{"nominal batch": mb,
         "miniBatchFraction": round(mb / train_size, 4),
         "median seconds": round(t, 4),
         "gradients / full batch": round(train_size / max(mb, 1), 1),
         "speedup over full batch": round(full / t, 2)}
        for mb, t in measured.items()]
print(pd.DataFrame(rows).to_string(index=False))
print("\nTimings are machine-dependent. The ratio in the last column is the point.")

 nominal batch  miniBatchFraction  median seconds  gradients / full batch  speedup over full batch
          8959             1.0000          0.0189                     1.0                     1.00
          4096             0.4572          0.0153                     2.2                     1.23
           512             0.0571          0.0136                    17.5                     1.39
            64             0.0071          0.0131                   140.0                     1.44

Timings are machine-dependent. The ratio in the last column is the point.


Compare the last two columns. Cutting the fraction from $1.0$ to $0.057$ divides the number of
gradient computations by about eighteen; it does not divide the time by anything close to
eighteen. Cutting it further to $0.007$ removes almost all of the remaining arithmetic and buys
almost nothing, because what is left is the scan and the fixed cost of scheduling a Spark job —
neither of which knows how large the batch was.

That is the whole of the guide's warning, and it is a statement about **this** workload rather than
about mini-batching in general. The per-example gradient here is one dot product and one
exponential over a two-element vector. It is cheap enough that the scan dominates.

---

## 4. When mini-batching does pay

The balance tips as soon as the per-example computation gets expensive. The `seqOp` below computes
exactly the same gradient and loss, but reaches them the long way round — a stand-in for a model
whose per-example work is genuinely heavy, such as a high-dimensional feature vector or a loss
requiring several transcendental evaluations per example.

In [6]:
def expensive_seq_op_for(bc_weights):
    # Arithmetically identical to the cheap version; deliberately many times the work,
    # standing in for a model whose per-example gradient is genuinely expensive.
    def seq_op(x, y):
        theta = np.dot(y[1], bc_weights)
        for _ in range(400):
            theta = np.arctan(np.tan(theta))       # a very slow way of leaving theta alone
        sigma = 1 / (np.exp(-theta) + 1)
        return (x[0] + y[1] * (-y[0] + sigma),
                x[1] + y[0] * (-theta) + np.log(1 + np.exp(theta)),
                x[2] + 1)
    return seq_op


np.random.seed(0)
b0 = np.random.normal(0, 0.1, len(traindata.first()[1]))
heavy = expensive_seq_op_for(b0)

measured_heavy = {mb: time_iterations(mb, repeats=3, seq_op=heavy)
                  for mb in (train_size, 4096, 512)}
full_heavy = measured_heavy[train_size]

rows = [{"nominal batch": mb,
         "miniBatchFraction": round(mb / train_size, 4),
         "median seconds": round(t, 4),
         "gradients / full batch": round(train_size / mb, 1),
         "speedup over full batch": round(full_heavy / t, 2)}
        for mb, t in measured_heavy.items()]
print(pd.DataFrame(rows).to_string(index=False))

 nominal batch  miniBatchFraction  median seconds  gradients / full batch  speedup over full batch
          8959             1.0000          0.1657                     1.0                     1.00
          4096             0.4572          0.0813                     2.2                     2.04
           512             0.0571          0.0193                    17.5                     8.57


Now the speedups track the fraction, because the arithmetic is the dominant term and the arithmetic
is what the sample removes. Nothing about Spark changed between this table and the previous one;
only the ratio of per-example work to per-record scan did.

The rule that falls out is worth stating in the form a reader can apply: **mini-batching in Spark
buys time in proportion to how much of an iteration is gradient evaluation rather than data
movement.** For a cheap loss it buys very little, and it costs gradient accuracy either way.

---

## 5. Why the accumulator carries a count

The accumulator in section 2 tracks a summed gradient, a summed loss, and a **count**, and divides
only at the end. The obvious simplification is to have each partition compute its own average
gradient and then average those. It is wrong, and the reason is the one property distributed
reduction depends on.

Addition is **associative** and has an identity, so a sum may be regrouped freely:
$(a+b)+c = a+(b+c)$. The arithmetic mean has neither property. The chapter's example is three
numbers grouped two ways:

In [7]:
mean = lambda *v: sum(v) / len(v)

print("mean(10, mean(30, 50)) =", mean(10, mean(30, 50)))
print("mean(mean(10, 30), 50) =", mean(mean(10, 30), 50))
print("the true mean of 10, 30, 50 is", mean(10, 30, 50))

mean(10, mean(30, 50)) = 25.0
mean(mean(10, 30), 50) = 35.0
the true mean of 10, 30, 50 is 30.0


Two groupings of the same three numbers, two different answers, and neither is the mean. Now the
same mistake in the shape it actually takes — an RDD whose partitions hold different numbers of
records, averaged per partition and then averaged again.

In [8]:
def partition_mean(it):
    # One mean per partition; empty partitions contribute nothing.
    v = list(it)
    return iter([sum(v) / len(v)]) if v else iter([])


def sum_and_count(rdd):
    return rdd.treeAggregate(
        (0.0, 0),
        lambda acc, v: (acc[0] + v, acc[1] + 1),        # seqOp: one record
        lambda a, b: (a[0] + b[0], a[1] + b[1]))        # combOp: associative


values = sc.parallelize([10.0, 30.0, 50.0], 2)
print("partition contents:", values.glom().collect())

naive = values.mapPartitions(partition_mean).mean()     # wrong: a mean of means
total, count = sum_and_count(values)                    # right: divide once, at the end
correct = total / count

print(f"\nmean of partition means : {naive}")
print(f"sum-and-count           : {correct}")
print(f"true mean               : {sum([10.0, 30.0, 50.0]) / 3}")
assert correct == 30.0

partition contents: [[10.0], [30.0, 50.0]]

mean of partition means : 25.0
sum-and-count           : 30.0
true mean               : 30.0


The last part is what makes this class of bug hard to catch. Re-partition the same data and the
answer moves — and, worse, sometimes stops being wrong:

In [9]:
rows = []
for p in (1, 2, 3, 4, 6):
    v = sc.parallelize([10.0, 30.0, 50.0], p)
    t, c = sum_and_count(v)
    rows.append({"partitions": p,
                 "layout": str(v.glom().collect()),
                 "mean of partition means": round(v.mapPartitions(partition_mean).mean(), 4),
                 "sum-and-count": round(t / c, 4)})
print(pd.DataFrame(rows).to_string(index=False))

 partitions                               layout  mean of partition means  sum-and-count
          1                 [[10.0, 30.0, 50.0]]                     30.0           30.0
          2               [[10.0], [30.0, 50.0]]                     25.0           30.0
          3             [[10.0], [30.0], [50.0]]                     30.0           30.0
          4         [[], [10.0], [30.0], [50.0]]                     30.0           30.0
          6 [[], [10.0], [], [30.0], [], [50.0]]                     30.0           30.0


Read the two right-hand columns against the layout. The mean of partition means is correct
whenever every non-empty partition holds the same number of records — which is the case for one
partition, and for three, four and six here — and wrong the moment they differ. So the naive
implementation passes on a single-partition laptop run, passes again on an evenly divided test
fixture, and fails on the cluster where the split is ragged. The sum-and-count column is 30 in
every row.
A gradient computed the wrong way is therefore correct on one partition, correct on a laptop where
everything lands in one partition anyway, and quietly wrong on the cluster — with no error, no
warning, and a training run that converges to a slightly different place depending on the machine
it ran on. That is Exercise 3, and it is why every accumulator in this chapter carries its count.

---

## Conclusion

An iteration of distributed gradient descent has two costs that scale differently. The gradient
arithmetic is proportional to the number of records sampled. The scan is proportional to the
number of records **stored**, because `sample` is a transformation that must inspect every record
in order to reject most of them. Reducing `miniBatchFraction` cuts the first and leaves the second
untouched, so the wall-clock saving is bounded by whatever share of the iteration the arithmetic
had to begin with — which on a cheap loss is small, and on an expensive one is most of it.

The accumulator that carries all this is the same triple everywhere in the chapter: a summed
gradient, a summed loss, and a count, merged by componentwise addition and divided exactly once, on
the driver. The count is not bookkeeping. It is what allows the average to be computed after the
reduction rather than during it, and therefore what makes the result independent of how the data
was partitioned.